In [7]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [8]:
import pandas as pd
from datetime import datetime, timedelta
from collections import defaultdict
import pytz

# Define time period
# Calculate the start date (2 days ago) at 00:00:00 UTC
start_date = (datetime.now(pytz.UTC) - timedelta(days=2)).date()

# Format as 'YYYY-MM-DDT00:00:00Z'
start = f"{start_date}T00:00:00Z"

# List of owners to pull from
import urllib.parse

list_of_owners = ['HTOC Org']
final_results = []
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR", "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition})'
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

         # Send the request
        response = tc.api_request(ro)

        # Parse response
        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")

    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean results
if final_results:
    # Extract and normalize data only if 'data' key exists and contains 'summary'
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')  # Remove duplicates based on 'indicator'
        observed_src = observed_src[observed_src['lastObserved'] >= start]
        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")
    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 275 unique observed indicators.


In [9]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,summary,...,text,hostName,dnsActive,whoisActive,address,sha1,source,url,Subject,indicator
10,6755399448033934,2025-05-27T13:19:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-27T17:56:02Z,3.0,80.0,142.93.13.29,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,142.93.13.29
86,4685354,2024-05-29T18:13:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2025-05-27T16:44:05Z,4.0,64.0,mavink.com,...,NaN,mavink.com,False,False,NaN,NaN,NaN,NaN,NaN,mavink.com
125,5629499542020488,2025-05-23T12:23:32Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-27T16:21:59Z,3.0,80.0,83.222.190.238,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.222.190.238
131,6755399448033933,2025-05-27T13:19:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-27T16:12:55Z,3.0,80.0,223.240.79.237,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,223.240.79.237
132,5629499542029727,2025-05-27T13:19:06Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-27T16:12:55Z,3.0,80.0,187.141.210.92,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,187.141.210.92
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9734,4553708,2024-03-29T13:28:05Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-23T06:13:45Z,3.0,14.0,64.64.112.156,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.64.112.156
9816,4554059,2024-03-29T14:24:45Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-23T06:13:44Z,3.0,12.0,156.146.63.170,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,156.146.63.170
9819,4553619,2024-03-29T13:13:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-23T06:13:44Z,3.0,5.0,169.150.196.153,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,169.150.196.153
9925,4546073,2024-03-18T18:16:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-23T06:13:43Z,5.0,32.0,195.181.160.173,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,195.181.160.173


In [10]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=3)

# Load observed data
observed_data_df = load_observed_data(file_paths)



Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250527.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250526.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20250525.csv']
Loaded data from 3 files.


In [11]:
import pandas as pd
from datetime import timedelta, date

# Define cutoff time in UTC
cutoff = pd.Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize an empty DataFrame to store filtered tags
filtered_tags = pd.DataFrame()

# Loop through each row in observed_src
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')
    
    if isinstance(tags_data, list):
        # Normalize the tags data for the current row
        tags = pd.json_normalize(tags_data)

        # Ensure 'name' column is of string type
        tags['name'] = tags['name'].astype(str)

        # Create a new column with a list of all tag names
        all_tags_list = tags['name'].tolist()

        # Filter for "API" tags
        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()

        if not api_tags.empty:
            # Add metadata columns and all_tags list as a single value (not as a series)
            for col in ['summary', 'observations', 'description', 'type', 'dateAdded', 'lastModified', 'lastObserved', 'webLink']:
                api_tags[col] = row.get(col)
            
            # Assign the all_tags list as a single value for the entire set of API tags
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)

            # Append to the filtered DataFrame
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Ensure 'lastObserved' is datetime
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Ensure necessary columns exist
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]

if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean the 'name' column by removing ' Splunk API'
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)

# Initialize the observed_date column with NaN
filtered_tags['observed_date'] = None

# Iterate through each row and search for matching indicator and opdiv
for index, row in filtered_tags.iterrows():
    # Extract summary and cleaned_name
    summary = row['summary']
    cleaned_name = row['cleaned_name']
    
    # Search for matching rows in the observed data
    match = observed_data_df[(observed_data_df['indicator'] == summary) & (observed_data_df['OpDiv'] == cleaned_name)]
    
    # If a match is found, extract the obs_date
    if not match.empty:
        # Assign the first matching obs_date
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')

# Drop the 'cleaned_name' column as it is no longer needed
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter to last 48 hours
recent_tags = filtered_tags[filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)].copy()

# Make `cutoff` a naive datetime to match the naive `observed_date`
cutoff_naive = cutoff.tz_convert(None)

# Apply the filter directly
recent_tags = filtered_tags[filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=2)].copy()
# Extract partner names and remove ' Splunk API' suffix
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

# Group partners per summary
partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))]).reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

# Merge partner information back into recent_tags
recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Apply additional filters
recent_tags = recent_tags[recent_tags['partner_count'] >= 2]
recent_tags = recent_tags[recent_tags['observations'] < 15000]

# Filter by dateAdded in the last 30 days
recent_tags = recent_tags[recent_tags['dateAdded'] >= date_added_cutoff]

# Drop duplicates based on 'summary'
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Drop unnecessary columns
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name'
]
recent_tags = recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], errors='ignore')


# Remove rows where 'all_tags' contains 'I&W'
recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x)]
recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'htoc_wl' in x)]


recent_tags


,id,lastUsed,summary,observations,description,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners
29,35760,2025-05-27T08:34:48Z,104.152.52.105,187,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:43+00:00,2025-05-27T08:35:54Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-26,2,"CMS, OS"
32,471298,2025-05-27T13:32:40Z,chaturbate.com,241,NaN,Stripped URL,2025-05-24 02:57:09+00:00,2025-05-27T08:35:54Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, HRSA Splunk API, NIH Splunk A...",2025-05-27,3,"DHA, HRSA, NIH"
43,35760,2025-05-27T08:34:48Z,104.152.52.238,295,FBI Email Alert May 14 2025 Medium IP IOCs,Address,2025-05-14 17:47:27+00:00,2025-05-27T08:35:54Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-26,3,"CMS, FDA, OS"
47,471298,2025-05-27T13:32:40Z,185.242.226.80,646,RITM0582833,Address,2025-05-21 13:34:43+00:00,2025-05-27T08:35:53Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, CMS Splunk API...",2025-05-26,2,"DHA, IHS"
60,35057,2025-05-27T08:35:15Z,104.152.52.242,299,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:50+00:00,2025-05-27T08:35:53Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-27,2,"CMS, FDA"
114,471298,2025-05-27T13:32:40Z,64.32.17.130,538,TACU_5-19-25,Address,2025-05-23 17:15:36+00:00,2025-05-27T08:35:51Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, VA OIS CSOC CTS Splunk, FDA S...",2025-05-27,3,"CMS, DHA, FDA"
117,35057,2025-05-27T08:35:15Z,104.152.52.139,226,FBI Email Alert May 14 2025 Medium IP IOCs,Address,2025-05-14 17:47:33+00:00,2025-05-27T08:35:51Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[VA OIS CSOC CTS Splunk, FDA Splunk API, suspi...",2025-05-27,3,"CMS, FDA, HRSA"
135,35760,2025-05-27T08:34:48Z,87.236.176.146,356,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:49+00:00,2025-05-27T08:35:50Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-27,2,"IHS, OS"
146,35057,2025-05-27T08:35:15Z,104.152.52.237,255,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:45+00:00,2025-05-27T08:35:50Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-05-27,3,"CMS, FDA, HRSA"
149,471298,2025-05-27T13:32:40Z,87.236.176.85,696,FBI Email Alert May 14 2025,Address,2025-05-14 17:44:44+00:00,2025-05-27T08:35:50Z,2025-05-27 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-05-26,3,"DHA, HRSA, OS"


import pandas as pd
import urllib.parse

# Extract unique indicators from recent_tags
indicators = recent_tags['summary'].unique()

# Initialize a list to store attribute data
attributes_data = []

# Track indicators with no attributes
indicators_with_no_attributes = []

# Iterate over unique indicators
for indicator in indicators:
    try:
        ro.set_http_method('GET')
        ro.set_request_uri(f'/v3/indicators/{indicator}?fields=attributes&resultStart=0&resultLimit=1000')
        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            data = response.json().get('data', {})
            attributes = data.get('attributes', {}).get('data', [])

            if not attributes:
                print(f"No attributes found for indicator: {indicator}")
                # Track indicators with no attributes
                indicators_with_no_attributes.append(indicator)
            else:
                for attr in attributes:
                    attr.update({
                        'id': data.get('id'),
                        'summary': data.get('summary'),
                        'type': data.get('type'),
                        'ownerName': data.get('ownerName')
                    })
                    attributes_data.append(attr)
        # Suppress non-JSON and known 400 error responses
    except Exception as e:
        # Suppress error output, including known 400 error
        if hasattr(e, 'response') and getattr(e.response, 'status_code', None) == 400:
            continue
        if "Status Code: 400" in str(e):
            continue
        pass

# Create attributes 
attributes_observed_src = pd.json_normalize(attributes_data)

# Un-nest 'createdBy' and filter out 'SOAR' entries
if not attributes_observed_src.empty and attributes_observed_src['createdBy.lastName'].notnull().any():
    attributes_observed_src = attributes_observed_src[attributes_observed_src['createdBy.lastName'] != 'SOAR']

# Drop duplicates based on 'id' to keep unique attribute records
attributes_observed_src = attributes_observed_src.drop_duplicates(subset='id').reset_index(drop=True)

# Filter recent_tags for indicators that had attributes
filtered_with_attrs = recent_tags[recent_tags['summary'].isin(attributes_observed_src['summary'])]

# Filter recent_tags for indicators that had no attributes
no_attrs_df = recent_tags[recent_tags['summary'].isin(indicators_with_no_attributes)]

# Combine both and remove duplicates based on 'summary'
filtered_recent_tags = pd.concat([filtered_with_attrs, no_attrs_df], ignore_index=True)
filtered_recent_tags = filtered_recent_tags.drop_duplicates(subset='summary').reset_index(drop=True)


In [ ]:
import pandas as pd
import requests
import time
import ipaddress
from datetime import datetime
import concurrent.futures

# API Keys
VT_API_KEY = "a8b3e24dbd2e6c0cb002784aeb8fee6217a6a425cb11ddf9a3d3361281fbbb08"
OTX_API_KEY = "ea83cf4792fc5db7acc741e82286c0b717fc99be98ec5b14de7e63cd3f74bcfe"

# Headers for API requests
VT_HEADERS = {"x-apikey": VT_API_KEY}
OTX_HEADERS = {"X-OTX-API-KEY": OTX_API_KEY}

# API URLs
VT_URL_IP = "https://www.virustotal.com/api/v3/ip_addresses/{}"
VT_URL_DOMAIN = "https://www.virustotal.com/api/v3/domains/{}"
OTX_URL_IP = "https://otx.alienvault.io/api/v1/indicators/IPv4/{}/general"
OTX_URL_DOMAIN = "https://otx.alienvault.io/api/v1/indicators/domain/{}/general"
OTX_URL_HOSTNAME = "https://otx.alienvault.io/api/v1/indicators/hostname/{}"

def is_ip(value):
    """ Check if the given value is a valid IP address. """
    try:
        ipaddress.ip_address(value)
        return True
    except ValueError:
        return False

def determine_query_type(query):
    """ Determine if the query is an IP, domain, or hostname. """
    if is_ip(query):
        return "ip"
    elif "." in query:
        return "hostname"
    else:
        return "domain"

def fetch_virustotal_data(query):
    """Fetch data from VirusTotal API for IP or Domain."""
    query_type = determine_query_type(query)
    url = VT_URL_IP.format(query) if query_type == "ip" else VT_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=VT_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json().get("data", {}).get("attributes", {})

        return {
            "search_term": query,
            "asn": data.get('asn'),
            "as_owner": data.get('as_owner'),
            "country": data.get('country'),
            "network": data.get('network'),
            "last_analysis_stats": data.get('last_analysis_stats'),
            "reputation": data.get('reputation'),
            "total_votes": data.get('total_votes'),
            "open_ports": [s.get("port") for s in data.get("services", []) if "port" in s],
            "link": f"https://www.virustotal.com/gui/ip-address/{query}" if query_type == "ip" else f"https://www.virustotal.com/gui/domain/{query}"
        }

    except Exception as e:
        print(f"VirusTotal Error for {query}: {e}")
        return None

def fetch_otx_data(query):
    """Fetch data from OTX API for IP, Domain, or Hostname."""
    query_type = determine_query_type(query)

    if query_type == "ip":
        url = OTX_URL_IP.format(query)
    elif query_type == "hostname":
        url = OTX_URL_HOSTNAME.format(query)
    else:
        url = OTX_URL_DOMAIN.format(query)

    try:
        response = requests.get(url, headers=OTX_HEADERS, verify=False)
        response.raise_for_status()
        data = response.json()

        return {
            "search_term": query,
            "base_indicator": data.get('base_indicator', {}),
            "reputation": data.get('reputation'),
            "geo": data.get('geo', {}),
            "whois": data.get('whois', {}),
            "open_ports": data.get('open_ports', []),
            "link": f"https://otx.alienvault.com/indicator/{query_type}/{query}"
        }

    except Exception as e:
        print(f"OTX Error for {query}: {e}")
        return None

def unnest_base_indicator(df):
    """ Unnest the 'base_indicator' column and create separate columns for its keys. """
    if 'base_indicator' not in df.columns:
        return df

    base_df = pd.json_normalize(df['base_indicator'])
    base_df.columns = [f"base_{col}" for col in base_df.columns]

    df = df.drop(columns=['base_indicator']).reset_index(drop=True)
    df = pd.concat([df, base_df], axis=1)
    return df

def process_indicator(indicator, observed_by, partner_count):
    """Fetch data for a single indicator."""
    timestamp = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    with concurrent.futures.ThreadPoolExecutor() as executor:
        vt_future = executor.submit(fetch_virustotal_data, indicator)
        otx_future = executor.submit(fetch_otx_data, indicator)

        vt_data = vt_future.result()
        otx_data = otx_future.result()

    if vt_data:
        vt_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    if otx_data:
        otx_data.update({
            "timestamp": timestamp,
            "observed_by": observed_by,
            "partner_count": partner_count
        })

    return vt_data, otx_data

def main(recent_tags):
    """ Main function to process indicators. """
    if 'summary' not in recent_tags.columns:
        print("The 'summary' column is missing.")
        return pd.DataFrame(), pd.DataFrame()

    search_terms = recent_tags['summary'].dropna().unique().tolist()
    print(f"Processing {len(search_terms)} unique search terms.")

    vt_results = []
    otx_results = []

    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = []
        for indicator in search_terms:
            partners = recent_tags.loc[recent_tags['summary'] == indicator, 'partners'].values
            partner_count = recent_tags.loc[recent_tags['summary'] == indicator, 'partner_count'].values
            observed_by = partners[0] if len(partners) > 0 else "N/A"
            partner_count = partner_count[0] if len(partner_count) > 0 else "N/A"

            futures.append(executor.submit(process_indicator, indicator, observed_by, partner_count))

        for future in concurrent.futures.as_completed(futures):
            vt_data, otx_data = future.result()
            if vt_data:
                vt_results.append(vt_data)
            if otx_data:
                otx_results.append(otx_data)

    vt_df = pd.DataFrame(vt_results)
    otx_df = pd.DataFrame(otx_results)

    otx_df = unnest_base_indicator(otx_df)

    return vt_df, otx_df

if __name__ == "__main__":
    vt_df, otx_df = main(recent_tags)
    print("Script completed successfully.")


Processing 0 unique search terms.
Script completed successfully.


In [24]:
vt_df

""


In [25]:
otx_df

""


In [26]:
import os
import pandas as pd
import docx
from datetime import datetime
from docx import Document
from docx.oxml import OxmlElement
from docx.oxml.ns import qn
from docx.oxml.shared import OxmlElement

# File paths
#TEMPLATE_PATH = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\I&W Report Template.docx"
TEMPLATE_PATH = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\I&W Report Template.docx"
#OUTPUT_DIR = r"z:\HTOC\HTOC Reports\I&W Reports\5. I&W Staging\Generated Reports"
OUTPUT_DIR = r"C:\Users\jaskew\Documents\project_repository\notebooks\I&W Reporting\Generated_reports"

# Ensure output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

def consolidate_sources(vt_df, otx_df):
    """ Consolidate links from both DataFrames for each search term. """
    # Collect links from VirusTotal
    vt_links = vt_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    vt_links.columns = ['search_term', 'vt_links']

    # Collect links from OTX
    otx_links = otx_df.groupby('search_term')['link'].apply(lambda x: ', '.join(x.dropna())).reset_index()
    otx_links.columns = ['search_term', 'otx_links']

    # Merge the two link sets
    consolidated = pd.merge(vt_links, otx_links, on='search_term', how='outer')

    # Combine the links, handling NaN values
    consolidated['sources'] = consolidated[['vt_links', 'otx_links']].fillna('').apply(
        lambda x: ', '.join(filter(None, x)), axis=1
    )

    return consolidated[['search_term', 'sources']]

def extract_date(timestamp):
    """ Extract only the date from the timestamp. """
    if pd.isna(timestamp) or timestamp == 'N/A':
        return 'N/A'
    
    # Handle datetime object or string
    try:
        # Attempt to parse as a datetime object
        if isinstance(timestamp, str):
            timestamp = datetime.strptime(timestamp, "%Y-%m-%d %H:%M:%S")
        return timestamp.strftime("%Y-%m-%d")
    except Exception:
        return 'N/A'

def populate_table(table, data):
    """ Populate a Word table with the given data. """
    # Iterate over data and populate rows
    for index, row in data.iterrows():
        # Insert a new row before the last row (template row)
        new_row = table.add_row().cells
        new_row[0].text = str(row.get('search_term', 'N/A'))
        new_row[1].text = str(row.get('type', 'N/A'))
        new_row[2].text = extract_date(row.get('observed_date', 'N/A'))
        # For the 'observed_by_otx' column, stack values instead of comma-separating
        # observed_by_otx: stack values (one per line) from OTX and VirusTotal "observed_by" columns if present
        observed_by_otx = ''
        observed_by_list = []

        # Try to get observed_by from OTX
        if 'observed_by_otx' in row and pd.notna(row['observed_by_otx']):
            observed_by_list.extend([v.strip() for v in str(row['observed_by_otx']).split(',') if v.strip()])
        elif 'observed_by' in row and pd.notna(row['observed_by']):
            observed_by_list.extend([v.strip() for v in str(row['observed_by']).split(',') if v.strip()])

        # Remove duplicates and join with newlines
        if observed_by_list:
            observed_by_otx = '\n'.join(sorted(set(observed_by_list)))
        else:
            observed_by_otx = 'N/A'
        new_row[3].text = str(observed_by_otx)
        new_row[4].text = str(row.get('notes', ''))
        
def fill_word_template(template_path, output_path, df):
    """ Fill the template with data and place sources outside the table. """
    if not os.path.exists(template_path):
        print(f"Template not found: {template_path}")
        return
    
    try:
        doc = Document(template_path)

        # Populate the table
        table = None
        for tbl in doc.tables:
            if "Indicators/Identifiers" in tbl.rows[0].cells[0].text:
                table = tbl
                break

        if table:
            populate_table(table, df)

        # Find and replace `{{sources}}` placeholder outside the table
        for para in doc.paragraphs:
            if "{{ipaddress}}" in para.text:
                # Try to get the first search_term as the IP address
                ip_address = str(df['search_term'].iloc[0]) if 'search_term' in df.columns else 'N/A'
                para.text = para.text.replace("{{ipaddress}}", ip_address)
            if "{{asn}}" in para.text:
                # Try to get ASN from vt_df if available, else use 'N/A'
                asn_value = str(df['asn'].iloc[0]) if 'asn' in df.columns and not df['asn'].isna().all() else 'N/A'
                para.text = para.text.replace("{{asn}}", asn_value)
            if "{{whois}}" in para.text:
                # Try to get WHOIS info from otx_df if available, else use 'N/A'
                whois_value = str(df['whois'].iloc[0]) if 'whois' in df.columns and not df['whois'].isna().all() else 'N/A'
                para.text = para.text.replace("{{whois}}", whois_value)
            if "{{partners}}" in para.text:
                # Try to get partners from recent_tags if available, else use 'N/A'
                partners_value = ''
                if 'search_term' in df.columns and not df.empty:
                    search_term = df['search_term'].iloc[0]
                    partners_row = recent_tags[recent_tags['summary'] == search_term]
                    if not partners_row.empty and 'partners' in partners_row.columns:
                        partners_value = str(partners_row['partners'].iloc[0])
                if not partners_value:
                    partners_value = 'N/A'
                para.text = para.text.replace("{{partners}}", partners_value)
            if "{{weblink}}" in para.text:
                weblink_value = ''
                # Try to get the first search_term as the indicator
                weblink_value = ''
                if 'search_term' in df.columns and not df.empty:
                    search_term = df['search_term'].iloc[0]
                    # Try to find a 'webLink' in df for the indicator (if present)
                    if 'webLink' in df.columns:
                        match = df[df['search_term'] == search_term]
                        if not match.empty and pd.notna(match['webLink'].iloc[0]):
                            weblink_value = str(match['webLink'].iloc[0])
                    # Fallback: try 'link' column if 'webLink' is not present or empty
                    if not weblink_value and 'link' in df.columns:
                        match = df[df['search_term'] == search_term]
                        if not match.empty and pd.notna(match['link'].iloc[0]):
                            weblink_value = str(match['link'].iloc[0])
                para.text = para.text.replace("{{weblink}}", "")
                if weblink_value:
                    # Add hyperlink using WordprocessingML (only show the link, no display text)
                    r_id = doc.part.relate_to(
                        weblink_value, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True
                    )
                    hyperlink = OxmlElement('w:hyperlink')
                    hyperlink.set(qn('r:id'), r_id)
                    new_run = OxmlElement('w:r')
                    rPr = OxmlElement('w:rPr')
                    rStyle = OxmlElement('w:rStyle')
                    rStyle.set(qn('w:val'), 'Hyperlink')
                    rPr.append(rStyle)
                    new_run.append(rPr)
                    t = OxmlElement('w:t')
                    t.text = weblink_value
                    new_run.append(t)
                    hyperlink.append(new_run)
                    para._p.append(hyperlink)
                else:
                    para.text = "N/A"
            if "{{sources}}" in para.text:
                # Build sources as hyperlinks if possible

                # Helper to add a hyperlink to a paragraph
                def add_hyperlink(paragraph, url):
                    # Create the w:hyperlink tag and add needed values
                    part = paragraph.part
                    r_id = part.relate_to(url, docx.opc.constants.RELATIONSHIP_TYPE.HYPERLINK, is_external=True)
                    hyperlink = OxmlElement('w:hyperlink')
                    hyperlink.set(qn('r:id'), r_id)
                    # Create a w:r element
                    new_run = OxmlElement('w:r')
                    # Create a w:rPr element
                    rPr = OxmlElement('w:rPr')
                    # Add color and underline for hyperlink style
                    rStyle = OxmlElement('w:rStyle')
                    rStyle.set(qn('w:val'), 'Hyperlink')
                    rPr.append(rStyle)
                    new_run.append(rPr)
                    # Create a w:t element and set the text
                    t = OxmlElement('w:t')
                    t.text = url
                    new_run.append(t)
                    hyperlink.append(new_run)
                    paragraph._p.append(hyperlink)

                # Remove the placeholder
                para.text = para.text.replace("{{sources}}", "")

                # Add each source as a hyperlink, stacked (one per line, no commas)
                sources = []
                for srcs in df['sources'].dropna().unique():
                    for src in [s.strip() for s in srcs.split(',') if s.strip()]:
                        sources.append(src)
                for i, src in enumerate(sources):
                    add_hyperlink(para, src)
                    if i < len(sources) - 1:
                        para.add_run().add_break()  # Add line break instead of comma

        # --- Save the document ---
        indicator_name = str(df['search_term'].iloc[0]) if 'search_term' in df.columns else 'Unnamed_Indicator'
        sanitized_name = indicator_name.replace(":", "_").replace("/", "_")

        current_date = datetime.now().strftime("%Y-%m-%d")
        folder_path = os.path.join(OUTPUT_DIR, current_date)
        os.makedirs(folder_path, exist_ok=True)

        output_path = os.path.join(folder_path, f"I&W_Report_{sanitized_name}.docx")
        doc.save(output_path)
        print(f"Saved report: {output_path}")

    except Exception as e:
        print(f"Error while generating report for {indicator_name}: {e}")

def main(vt_df, otx_df, recent_tags):
    """ Main function to generate one I&W report per indicator. """
    # Combine vt_df and otx_df on 'search_term'
    combined_df = pd.merge(vt_df, otx_df, on='search_term', how='outer', suffixes=('_vt', '_otx'))

    # Consolidate sources into a single column
    sources_df = consolidate_sources(vt_df, otx_df)
    combined_df = pd.merge(combined_df, sources_df, on='search_term', how='left')

    # Merge recent_tags and drop 'summary' to avoid duplication
    if not recent_tags.empty:
        combined_df = pd.merge(
            combined_df,
            recent_tags[['summary', 'observations', 'type', 'partners', 'observed_date', 'webLink']],
            left_on='search_term',
            right_on='summary',
            how='left'
        )
        combined_df.drop(columns=['summary'], inplace=True)
        display(combined_df)
    # Generate individual reports per unique indicator
    current_date = pd.Timestamp.now().strftime("%Y-%m-%d")
    for indicator in combined_df['search_term'].unique():
        indicator_df = combined_df[combined_df['search_term'] == indicator]
        sanitized_indicator = indicator.replace(":", "_").replace("/", "_")
        output_file = os.path.join(OUTPUT_DIR, f"I&W_Report_{sanitized_indicator}_{current_date}.docx")
        #fill_word_template(TEMPLATE_PATH, output_file, indicator_df)
    
if __name__ == "__main__":
    main(vt_df, otx_df, recent_tags)



KeyError: 'search_term'